# 如何在子链之间进行路由

- 有两种方式可以执行路由：
    - 从 RunnableLambda 有条件地返回可运行组件（推荐）
    - 使用 RunnableBranch (遗留版)

In [2]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")
model = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)

SECESSFULLY!


# 示例设置
让我们创建一个链，以识别传入的问题是关于 *LangChain、Deepseek* 还是其他

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = (
    PromptTemplate.from_template(
        """Given the user question below, classify it as either being about `LangChain`, `Deepseek`, or `Other`.

Do not respond with more than one word.

<question>
{question}
</question>

Classification:"""
    )
    | ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)
    | StrOutputParser()
)

In [ ]:
for chunck in prompt.stream({"question": "how do I call Anthropic?"}):
    print(chunck, end="", flush=True )

Other

In [5]:
langchain_chain = PromptTemplate.from_template(
    """You are an expert in langchain. \
Always answer questions starting with "As langchain told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | model

deepseek_chain = PromptTemplate.from_template(
    """You are an expert in deepseek. \
Always answer questions starting with "As deepseek told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | model

general_chain = PromptTemplate.from_template(
    """Respond to the following question:

Question: {question}
Answer:"""
) | model

### 使用自定义函数（推荐）

In [6]:
def route(info):
    if "deepseek" in info["topic"].lower():
        return deepseek_chain
    if "langchain" in info["topic"].lower():
        return langchain_chain
    else:
        return general_chain

In [ ]:
from langchain_core.runnables import RunnableLambda

full_chain = {"topic": prompt, "question": lambda x: x["question"]} | RunnableLambda(
    route
)

In [9]:
for chunck in  full_chain.stream({"question" : "我该怎么使用langchain？"}):
    print(chunck.content, end="", flush=True)

As langchain told me, 要使用LangChain，您可以按照以下步骤进行：

1. **安装LangChain库**  
   使用pip或conda安装：  
   ```bash
   pip install langchain
   # 或安装包含可选依赖的版本
   pip install langchain[all]
   ```

2. **配置环境变量**  
   设置API密钥（如OpenAI、Azure等）：  
   ```bash
   export OPENAI_API_KEY="your-key"
   ```

3. **核心模块使用示例**  
   - **LLM调用**：  
     ```python
     from langchain.llms import OpenAI
     llm = OpenAI(temperature=0.7)
     print(llm("请解释量子计算"))
     ```

   - **提示模板**：  
     ```python
     from langchain.prompts import PromptTemplate
     prompt = PromptTemplate(input_variables=["topic"], template="请解释{topic}")
     formatted_prompt = prompt.format(topic="人工智能")
     ```

   - **链式操作**：  
     ```python
     from langchain.chains import LLMChain
     chain = LLMChain(llm=llm, prompt=prompt)
     print(chain.run("区块链"))
     ```

   - **代理（Agents）**：  
     ```python
     from langchain.agents import load_tools, initialize_agent
     tools = load_tools(["serpapi"], llm=llm)
     agent = initialize_

In [10]:
for chunck in  full_chain.stream({"question" : "我该怎么使用deepseek？"}):
    print(chunck.content, end="", flush=True)

As deepseek told me, 使用DeepSeek非常简单便捷！以下是主要使用方式：

🌐 **网页版使用**
- 直接访问DeepSeek官网，在浏览器中即可开始对话
- 无需下载安装，打开就能用

📱 **移动端App**
- 可以通过官方应用商店下载DeepSeek App
- 随时随地都能使用，更加方便

💡 **核心功能**
- 直接输入问题即可获得回答
- 支持上传文件（图像、txt、pdf、ppt、word、excel等）
- 需要时点击联网搜索按钮获取最新信息
- 128K的上下文长度，能记住很长的对话内容

🎯 **使用技巧**
- 问题描述越详细，回答越精准
- 可以多轮对话，我会记住之前的内容
- 完全免费使用，无需担心费用问题

有什么具体想了解的功能，我很乐意为您详细介绍！😊

In [13]:
for chunck in  full_chain.stream({"question" : "疾如风，徐如林，掠如火，难知如因！"}):
    print(chunck.content, end="", flush=True)

您引用的是《孙子兵法》军争篇中的名句，但原文最后一句通常是“**不动如山**”，您写的“难知如因”可能是“**难知如阴**”的笔误。这句话完整的正确版本是：

**“故其疾如风，其徐如林，侵掠如火，不动如山，难知如阴，动如雷震。”**

**含义解析：**
1. **疾如风**：军队行动迅速时，如狂风般迅捷。  
2. **徐如林**：行军缓慢时，如严整的森林般肃穆有序。  
3. **侵掠如火**：进攻时如烈火般猛烈，势不可挡。  
4. **不动如山**：防守时如山岳般稳固，不可动摇。  
5. **难知如阴**：隐蔽时如阴云蔽日，深不可测。  
6. **动如雷震**：发动攻势时如雷霆万钧，猝不及防。

**背景与影响：**  
这句话概括了古代战争中军队应具备的战术素养与精神状态，后被日本战国名将**武田信玄**提炼为“风林火山”（取自前四句），成为其军旗标识，广为人知。

若您希望探讨其他版本或具体应用场景，可以补充说明～ 🌟

### 使用 RunnableBranch（遗留版）

In [ ]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "deepseek" in x["topic"].lower(), deepseek_chain),
    (lambda x: "langchain" in x["topic"].lower(), langchain_chain),
    general_chain,
)
full_chain = {"topic": prompt, "question": lambda x: x["question"]} | branch

for chunck in full_chain.stream({"question": "怎么使用 deepseek 的 function call?"}):
    print(chunck.content, end="", flush=True)

As deepseek told me, 使用DeepSeek的function call功能主要分为以下几个步骤：

1. **准备阶段**：
   - 在API请求中定义好可用的工具函数列表
   - 每个函数需要包含名称、描述和参数schema

2. **调用方式**：
   - 在对话消息中设置`tools`参数来声明可用函数
   - 当模型识别到需要调用函数时，会在响应中返回`tool_calls`字段

3. **具体流程**：
   - 用户发送包含函数调用需求的查询
   - 模型分析后决定是否以及如何调用函数
   - 如果调用函数，模型会返回函数名称和参数
   - 开发者执行相应函数并返回结果
   - 将函数执行结果再次发送给模型进行后续处理

4. **示例代码结构**：
```python
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "查询北京的天气"}],
    tools=[{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取城市天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                }
            }
        }
    }]
)
```

建议查阅DeepSeek官方文档获取最新的详细使用指南和代码示例。

## 通过语义相似性进行路由
一种特别有用的技术是使用嵌入将查询路由到最相关的提示。

In [ ]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_community.embeddings import HuggingFaceEmbeddings

physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

embeddings = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)


physics_and_math_chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | model
    | StrOutputParser()
)


In [19]:
for chuck in physics_and_math_chain.stream("怎么理解麦克斯韦电磁场是一种特殊的规范场？"):
    print(chuck, end="", flush=True)

Using PHYSICS
好，我们一步步来理解这个问题。  

---

## 1. 电磁场的规范对称性

在经典电动力学中，麦克斯韦方程组可以写成用四维势 \(A_\mu = (\phi, \mathbf{A})\) 表示的形式，其中  
\[
\mathbf{E} = -\nabla \phi - \frac{\partial \mathbf{A}}{\partial t}, \quad \mathbf{B} = \nabla \times \mathbf{A}.
\]  
这里 \(A_\mu\) 不是唯一确定的：如果我们做变换  
\[
A_\mu \to A_\mu + \partial_\mu \lambda
\]  
（其中 \(\lambda(x,t)\) 是任意标量函数），对应的 \(\mathbf{E}\) 和 \(\mathbf{B}\) 不变。  
这就是 **U(1) 规范对称性**，是阿贝尔的（因为变换可交换）。

---

## 2. 规范场的普遍概念

在杨振宁–米尔斯 1954 年的理论中，“规范场”一般指一个主纤维丛上的联络（connection），其规范群可以是任意李群（如 \(SU(N)\)）。  
规范势 \(A_\mu\) 取值在群的李代数中，场强  
\[
F_{\mu\nu} = \partial_\mu A_\nu - \partial_\nu A_\mu + [A_\mu, A_\nu]
\]  
对于非阿贝尔群，后面有对易子项，场强不是规范不变的，而是在伴随表示下变换。

---

## 3. 麦克斯韦场作为特殊的规范场

麦克斯韦电磁场对应的规范群是 **U(1)**，这是一个阿贝尔（可交换）群，且是一维的。  
所以它的李代数是 \(\mathbb{R}\)（或 \(i\mathbb{R}\)），对易子 \([A_\mu, A_\nu] = 0\)。  
于是场强为  
\[
F_{\mu\nu} = \partial_\mu A_\nu - \partial_\nu A_\mu
\]  
这就是电磁场张量。

因此，麦克斯韦理论是**规范群为 U(1) 的规范场**，是**阿贝尔规范场**的最简单例子，也是历史上最早的规范理论（虽然当时不叫这个名称）。

---

## 4. 为什么说“特殊”

- **

In [22]:
for chuck in physics_and_math_chain.stream("怎么理解在同调代数里的的snake lemma？"):
    print(chuck, end="", flush=True)

Using MATH
好的，我们先一步步来拆解这个问题。  

---

## 1. 问题定位

“蛇引理”（Snake lemma）是同调代数里的一个基本引理，它描述了在**交换图**的某些行正合、列正合的情况下，如何构造一个长正合序列。  
它通常出现在讨论链复形的短正合序列与它们的长正合同调序列之间的关系时。

---

## 2. 蛇引理的典型形式

假设我们有如下交换图，其中行是正合的：

\[
\begin{matrix}
 &  & A & \xrightarrow{f} & B & \xrightarrow{g} & C & \to & 0 \\
 &  & \downarrow a &  & \downarrow b &  & \downarrow c &  & \\
 0 & \to & A' & \xrightarrow{f'} & B' & \xrightarrow{g'} & C' &  & 
\end{matrix}
\]

这里 \(a, b, c\) 是某个 Abel 范畴（比如 \(R\)-模）中的态射。

---

## 3. 结论

蛇引理断言存在一个**连接同态**（connecting homomorphism）

\[
\delta : \ker c \to \operatorname{coker} a
\]

使得下面的序列是正合的：

\[
\ker a \to \ker b \to \ker c \xrightarrow{\delta} \operatorname{coker} a \to \operatorname{coker} b \to \operatorname{coker} c
\]

其中前两个映射由 \(f, g\) 诱导，后两个映射由 \(f', g'\) 诱导。

---

## 4. 直观理解

1. **图的来源**  
   常见情形：链复形的短正合序列  
   \[
   0 \to X_\bullet \to Y_\bullet \to Z_\bullet \to 0
   \]
   对每个 \(n\)，取 \(A = Z_n\), \(B = Y_n\), \(C = Z_n\)，而 \(A' = X_{n-1}\) 等（实际上更常见的是用 \(A = X_n\) 等，但连接映射会降